In [99]:
import pandas as pd
import altair as alt
import requests
import json

In [2]:
url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/licenses_fall2022.csv"
df = pd.read_csv(url)

In [3]:
df.head()


,_id,License Type,Description,License Number,License Status,Business,Title,First Name,Middle,Last Name,...,Specialty/Qualifier,Controlled Substance Schedule,Delegated Controlled Substance Schedule,Ever Disciplined,LastModifiedDate,Case Number,Action,Discipline Start Date,Discipline End Date,Discipline Reason
0,1189509,DETECTIVE BOARD,PERMANENT EMPLOYEE REGISTRATION,129446286,NOT RENEWED,N,NaN,EILEEN,NaN,SANTACRUZ,...,NaN,NaN,NaN,N,03/18/2022,NaN,NaN,NaN,NaN,NaN
1,801037,DETECTIVE BOARD,FIREARM CONTROL CARD,229030294.0,NOT RENEWED,N,NaN,DAGMAR,J,NORDLUND,...,NaN,NaN,NaN,N,08/16/2006,NaN,NaN,NaN,NaN,NaN
2,365129,COSMO,LICENSED COSMETOLOGIST,11053076.0,NOT RENEWED,N,NaN,RADOJE,NaN,ZELENOVIC,...,NaN,NaN,NaN,N,05/26/2006,NaN,NaN,NaN,NaN,NaN
3,595427,COSMO,LICENSED COSMETOLOGIST,11295645.0,ACTIVE,N,NaN,BECKY SUE,L,BURROUGHS,...,NaN,NaN,NaN,N,11/12/2021,NaN,NaN,NaN,NaN,NaN
4,653668,COSMO,LICENSED NAIL TECHNICIAN,169006247,NOT RENEWED,N,NaN,BILL G,L,LETNER,...,NaN,NaN,NaN,N,05/30/2006,NaN,NaN,NaN,NaN,NaN


In [4]:
df.info

<bound method DataFrame.info of           _id     License Type                      Description  \
0     1189509  DETECTIVE BOARD  PERMANENT EMPLOYEE REGISTRATION   
1      801037  DETECTIVE BOARD             FIREARM CONTROL CARD   
2      365129            COSMO           LICENSED COSMETOLOGIST   
3      595427            COSMO           LICENSED COSMETOLOGIST   
4      653668            COSMO         LICENSED NAIL TECHNICIAN   
...       ...              ...                              ...   
9995   888281  DETECTIVE BOARD  PERMANENT EMPLOYEE REGISTRATION   
9996   766623  DETECTIVE BOARD             FIREARM CONTROL CARD   
9997   399398            COSMO           LICENSED COSMETOLOGIST   
9998   486713            COSMO           LICENSED COSMETOLOGIST   
9999   744770           DENTAL      REGISTERED DENTAL HYGIENIST   

     License Number            License Status Business Title      First Name  \
0         129446286               NOT RENEWED        N   NaN          EILEEN   
1  

In [5]:
na_counts = df.isna().sum()
na_counts = na_counts[na_counts > 0]
na_counts

License Number                                60
Title                                       9890
First Name                                   395
Middle                                      6378
Last Name                                    395
Prefix                                      9997
Suffix                                      9590
BusinessDBA                                 9885
Original Issue Date                            5
Effective Date                               792
Expiration Date                              500
City                                          11
Zip                                           71
County                                       411
Specialty/Qualifier                         9692
Controlled Substance Schedule               9791
Delegated Controlled Substance Schedule    10000
Case Number                                 9657
Action                                      9658
Discipline Start Date                       9657
Discipline End Date 

In [6]:
threshold = 0.90  
license_df = df.drop(columns=df.columns[df.isna().mean() > threshold])
license_df.head()

,_id,License Type,Description,License Number,License Status,Business,First Name,Middle,Last Name,Business Name,Original Issue Date,Effective Date,Expiration Date,City,State,Zip,County,Ever Disciplined,LastModifiedDate
0,1189509,DETECTIVE BOARD,PERMANENT EMPLOYEE REGISTRATION,129446286,NOT RENEWED,N,EILEEN,NaN,SANTACRUZ,SHAQUITA MCKETHAN,02/03/2020,02/03/2020,09/30/2021,CHICAGO,IL,60617.0,COOK,N,03/18/2022
1,801037,DETECTIVE BOARD,FIREARM CONTROL CARD,229030294.0,NOT RENEWED,N,DAGMAR,J,NORDLUND,EDWARD J HERDRICH,02/07/1995,02/07/1995,12/31/2003,ELGIN,IL,60121.0,KANE,N,08/16/2006
2,365129,COSMO,LICENSED COSMETOLOGIST,11053076.0,NOT RENEWED,N,RADOJE,NaN,ZELENOVIC,ELEANOR L KUKLA,02/28/1945,02/28/1945,09/30/1983,CHICAGO,IL,60618.0,COOK,N,05/26/2006
3,595427,COSMO,LICENSED COSMETOLOGIST,11295645.0,ACTIVE,N,BECKY SUE,L,BURROUGHS,AMBER L BARBA,11/22/2011,11/12/2021,09/30/2023,SCHAUMBURG,IL,60173.0,DUPAGE,N,11/12/2021
4,653668,COSMO,LICENSED NAIL TECHNICIAN,169006247,NOT RENEWED,N,BILL G,L,LETNER,CHERYL L SCHULZE,07/12/1995,07/12/1995,10/31/2002,CHICAGO,IL,60634,COOK,N,05/30/2006


In [7]:
license_df.isna().sum()

_id                       0
License Type              0
Description               0
License Number           60
License Status            0
Business                  0
First Name              395
Middle                 6378
Last Name               395
Business Name             0
Original Issue Date       5
Effective Date          792
Expiration Date         500
City                     11
State                     0
Zip                      71
County                  411
Ever Disciplined          0
LastModifiedDate          0
dtype: int64

In [8]:
necessary_cols = [
    '_id',
    'License Number',
    'License Type',
    'Description',
    'License Status',
    'Business',
    'Business Name',
    'Original Issue Date',
    'Effective Date',
    'Expiration Date',
    'LastModifiedDate',
    'City',
    'State',
    'Zip',
    'County',
    'Ever Disciplined'
]

license_df = license_df[necessary_cols]
license_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   _id                  10000 non-null  int64 
 1   License Number       9940 non-null   object
 2   License Type         10000 non-null  object
 3   Description          10000 non-null  object
 4   License Status       10000 non-null  object
 5   Business             10000 non-null  object
 6   Business Name        10000 non-null  object
 7   Original Issue Date  9995 non-null   object
 8   Effective Date       9208 non-null   object
 9   Expiration Date      9500 non-null   object
 10  LastModifiedDate     10000 non-null  object
 11  City                 9989 non-null   object
 12  State                10000 non-null  object
 13  Zip                  9929 non-null   object
 14  County               9589 non-null   object
 15  Ever Disciplined     10000 non-null  object
dtypes: in

In [57]:
license_df.info

<bound method DataFrame.info of           _id License Number     License Type  \
0     1189509      129446286  DETECTIVE BOARD   
1      801037    229030294.0  DETECTIVE BOARD   
2      365129     11053076.0            COSMO   
3      595427     11295645.0            COSMO   
4      653668      169006247            COSMO   
...       ...            ...              ...   
9995   888281    129002843.0  DETECTIVE BOARD   
9996   766623      229014180  DETECTIVE BOARD   
9997   399398       11120249            COSMO   
9998   486713       11193270            COSMO   
9999   744770      020012709           DENTAL   

                          Description            License Status Business  \
0     PERMANENT EMPLOYEE REGISTRATION               NOT RENEWED        N   
1                FIREARM CONTROL CARD               NOT RENEWED        N   
2              LICENSED COSMETOLOGIST               NOT RENEWED        N   
3              LICENSED COSMETOLOGIST                    ACTIVE        N   

In [97]:
license_df["State"] = license_df["State"].astype(str).str.strip().str.upper()
license_df["County"] = license_df["County"].astype(str).str.strip()
license_df["License Type"] = license_df["License Type"].astype(str).str.strip()
license_il = license_df[license_df["State"] == "IL"].copy()

In [98]:
county_counts = (
    license_il.groupby("County")
    .size()
    .reset_index(name="Count")
)
county_counts["County_upper"] = county_counts["County"].str.upper()
county_counts

,County,Count,County_upper
0,ADAMS,41,ADAMS
1,ALEXANDER,2,ALEXANDER
2,BOND,4,BOND
3,BOONE,27,BOONE
4,BROWN,2,BROWN
...,...,...,...
105,WILL,391,WILL
106,WILLIAMSON,35,WILLIAMSON
107,WINNEBAGO,167,WINNEBAGO
108,WOODFORD,24,WOODFORD


In [88]:
topo_url = "https://cdn.jsdelivr.net/npm/us-atlas@3/counties-10m.json"
counties = alt.topo_feature(topo_url, "counties")


In [89]:
raw_topo = requests.get(topo_url).json()

geo_lookup = {}
for g in raw_topo["objects"]["counties"]["geometries"]:
    county_name = g["properties"].get("name", "").upper()
    geo_lookup[county_name] = g["id"]


county_counts["fips"] = county_counts["County_upper"].map(geo_lookup)


In [93]:
outline = (
    alt.Chart(counties)
    .mark_geoshape(
        fill="white",      
        stroke="black",     
        strokeWidth=0.5
    )
    .transform_filter("substring(datum.id, 0, 2) == '17'")
    .project("mercator")
)

filled = (
    alt.Chart(counties)
    .mark_geoshape(
        stroke="black",   
        strokeWidth=0.5
    )
    .encode(
        color=alt.Color("Count:Q", scale=alt.Scale(scheme="blues")),
        tooltip=["County:N", "Count:Q"]
    )
    .transform_lookup(
        lookup="id",
        from_=alt.LookupData(
            county_counts,
            key="fips",
            fields=["County", "Count"]
        )
    )
    .transform_filter("substring(datum.id, 0, 2) == '17'")
    .project("mercator")
)

chart2 = (outline + filled).properties(
    width=500,
    height=600,
    title="Illinois Licenses by County"
)

chart2

alt.LayerChart(...)

In [94]:

license_df["County"] = license_df["County"].astype(str).str.strip()
license_df["License Type"] = license_df["License Type"].astype(str).str.strip()


license_il = license_df[license_df["State"] == "IL"].copy()

county_type_counts = (
    license_il.groupby(["County", "License Type"])
    .size()
    .reset_index(name="Count")
)

In [95]:
county_param = alt.param(
    name="County",
    bind=alt.binding_select(
        options=sorted(county_type_counts["County"].unique()),
        name="Select County: "
    ),
    value=sorted(county_type_counts["County"].unique())[0]  # default
)


In [96]:
interactive_top5 = (
    alt.Chart(county_type_counts)
    .transform_filter("datum.County == County")  # param comparison
    .transform_window(
        rank="rank(Count)",
        sort=[alt.SortField("Count", order="descending")],
        groupby=["County"]
    )
    .transform_filter("datum.rank <= 5")  # Top 5
    .mark_bar()
    .encode(
        x=alt.X("License Type:N", sort="-y", title="License Type"),
        y=alt.Y("Count:Q", title="Number of Licenses"),
        color="License Type:N",
        tooltip=["County:N", "License Type:N", "Count:Q"]
    )
    .add_params(county_param)
    .properties(
        width=500,
        height=400,
        title="Top 5 License Types by Selected County (Illinois)"
    )
)

interactive_top5


alt.Chart(...)

In [81]:
(chart2 | interactive_top5)

alt.HConcatChart(...)

In [100]:
chart2.save("chart1_top_license_types.html")


In [101]:
interactive_top5.save("chart2_illinois_map.html")